# Notebook 07 — Series Temporales NDVI y NDRE (2018–2024)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-03/colab/07_series_temporales_ndvi.ipynb)

## Maestría en Ingeniería — Universidad del Magdalena
### Sesión 3: Análisis multitemporal sobre la zona cacaotera de la SNSM

---

**Objetivo:** Construir series temporales de NDVI y NDRE sobre la Sierra Nevada de Santa Marta (2018–2024) usando Google Earth Engine desde Python. Al finalizar deberás:
- Visualizar cómo cambia el NDVI año a año
- Comparar NDVI vs NDRE en dosel denso
- Detectar tendencias con el test Mann-Kendall
- Exportar resultados como CSV y GeoTIFF

**Tiempo estimado:** 45–60 minutos

**Referencias:**
- Tucker, C.J. (1979). Red and photographic infrared combinations. *RSE*, 8(2). DOI: 10.1016/0034-4257(79)90013-0
- Huete, A. et al. (2002). Overview of MODIS vegetation indices. *RSE*, 83(1–2). DOI: 10.1016/S0034-4257(02)00096-2
- Mann, H.B. (1945). Nonparametric tests against trend. *Econometrica*, 13, 245.

In [ ]:
# Instalar paquetes (solo primera vez en Colab)
!pip install earthengine-api geemap pymannkendall -q

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pymannkendall as mk

# Autenticar Google Earth Engine
try:
    ee.Initialize(project='teledeteccion-miguepoloc')  # Cambia por tu proyecto GEE
except Exception:
    ee.Authenticate()
    ee.Initialize(project='teledeteccion-miguepoloc')

print('Google Earth Engine inicializado')

## Paso 1 — Área de estudio: zona cacaotera SNSM

In [ ]:
aoi = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

Map = geemap.Map(center=[10.75, -74.0], zoom=10)
Map.addLayer(aoi, {'color': 'red'}, 'Zona cacaotera SNSM')
Map

## Paso 2 — Cargar Sentinel-2 y calcular índices

> **¿Por qué temporada seca (enero–abril)?**
> Comparar la misma estación cada año elimina la variabilidad estacional.
> Si el NDVI sube de un año a otro, es cambio real, no el efecto de las lluvias.

In [ ]:
def mascara_nubes(img):
    """Aplica máscara SCL: excluye sombras (3), nubes media (8), alta (9) y cirros (10)."""
    scl = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(mask).divide(10000)

def agregar_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndre = img.normalizedDifference(['B8', 'B5']).rename('NDRE')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {'NIR': img.select('B8'), 'RED': img.select('B4'), 'BLUE': img.select('B2')}
    ).rename('EVI')
    ndmi = img.normalizedDifference(['B8', 'B11']).rename('NDMI')
    return img.addBands([ndvi, ndre, evi, ndmi])

coleccion = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filter(ee.Filter.calendarRange(1, 4, 'month'))   # Enero-Abril
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(mascara_nubes)
    .map(agregar_indices)
)

n = coleccion.size().getInfo()
print(f'Imágenes disponibles: {n}')

## Paso 3 — Composición anual por mediana (2018–2024)

In [ ]:
years = list(range(2018, 2025))

def composicion_anual(year):
    return (
        coleccion
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .median()
        .set('year', year)
        .clip(aoi)
    )

composiciones = ee.ImageCollection(ee.List([composicion_anual(y) for y in years]))

# Comparar NDVI 2018 vs 2024
viz = {'bands': ['NDVI'], 'min': 0.1, 'max': 0.85,
       'palette': ['#d73027','#fee090','#91cf60','#1a9850']}

Map2 = geemap.Map(center=[10.75, -74.0], zoom=10)
Map2.addLayer(composicion_anual(2018), viz, 'NDVI 2018')
Map2.addLayer(composicion_anual(2024), viz, 'NDVI 2024')
Map2

## Paso 4 — Diferencia de cambio NDVI (2024 - 2018)

> Verde/azul = aumento de vegetación | Rojo = pérdida de vegetación

In [ ]:
img_2018 = composicion_anual(2018)
img_2024 = composicion_anual(2024)
diff = img_2024.select('NDVI').subtract(img_2018.select('NDVI')).rename('dNDVI')

Map3 = geemap.Map(center=[10.75, -74.0], zoom=10)
Map3.addLayer(diff, {'min': -0.3, 'max': 0.3,
    'palette': ['#d73027','#fdae61','#ffffbf','#a6d96a','#1a9850']}, 'Cambio NDVI')
Map3

## Paso 5 — Extraer estadísticas por año

In [ ]:
def extraer_stats(img):
    stats = img.select(['NDVI','NDRE','EVI','NDMI']).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi, scale=10, maxPixels=1e9
    )
    return ee.Feature(None, stats.set('year', img.get('year')))

features = composiciones.map(extraer_stats).getInfo()['features']

datos = []
for f in features:
    p = f['properties']
    datos.append({
        'year': int(p.get('year', 0)),
        'NDVI_mean': p.get('NDVI_mean'),
        'NDVI_std': p.get('NDVI_stdDev'),
        'NDRE_mean': p.get('NDRE_mean'),
        'NDRE_std': p.get('NDRE_stdDev'),
        'EVI_mean': p.get('EVI_mean'),
        'NDMI_mean': p.get('NDMI_mean'),
    })

df = pd.DataFrame(datos).sort_values('year').dropna()
print(df.to_string(index=False))

## Paso 6 — Visualización: NDVI vs NDRE a lo largo del tiempo

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# NDVI
ax1.plot(df['year'], df['NDVI_mean'], 'o-', color='#2d6a4f', lw=2.5, ms=8, label='NDVI')
ax1.fill_between(df['year'], df['NDVI_mean']-df['NDVI_std'],
                 df['NDVI_mean']+df['NDVI_std'], alpha=0.2, color='#2d6a4f')
ax1.axhline(0.6, color='gray', ls='--', alpha=0.6, label='Dosel denso (0.6)')
ax1.set_ylabel('NDVI medio', fontsize=12)
ax1.set_title('Series Temporales NDVI y NDRE — Zona Cacaotera SNSM\n(Sentinel-2, enero–abril, 2018–2024)',
              fontsize=14, fontweight='bold')
ax1.legend(); ax1.set_ylim(0, 1); ax1.grid(alpha=0.3)

# NDRE vs NDVI
ax2.plot(df['year'], df['NDRE_mean'], 's-', color='#e63946', lw=2.5, ms=8, label='NDRE')
ax2.fill_between(df['year'], df['NDRE_mean']-df['NDRE_std'],
                 df['NDRE_mean']+df['NDRE_std'], alpha=0.2, color='#e63946')
ax2.plot(df['year'], df['NDVI_mean'], 'o--', color='#2d6a4f', lw=1.5, alpha=0.6, ms=5, label='NDVI (ref.)')
ax2.set_xlabel('Año', fontsize=12)
ax2.set_ylabel('Índice medio', fontsize=12)
ax2.set_title('NDRE vs NDVI: sensibilidad a clorofila y detección de estrés temprano', fontsize=13)
ax2.legend(); ax2.set_ylim(0, 1); ax2.grid(alpha=0.3)
ax2.set_xticks(df['year'].values)

plt.tight_layout()
plt.savefig('series_temporales_ndvi_ndre_snsm.png', dpi=150, bbox_inches='tight')
plt.show()

## Paso 7 — Test Mann-Kendall: ¿hay una tendencia significativa?

> **p-value < 0.05** → tendencia estadísticamente significativa  
> **Pendiente de Sen** → tasa de cambio por año (NDVI/año)

In [ ]:
res_ndvi = mk.original_test(df['NDVI_mean'].values)
res_ndre = mk.original_test(df['NDRE_mean'].values)

print('TEST MANN-KENDALL — Zona Cacaotera SNSM (2018–2024)')
print('=' * 55)
for nombre, res in [('NDVI', res_ndvi), ('NDRE', res_ndre)]:
    sig = '(SIGNIFICATIVA p<0.05)' if res.p < 0.05 else '(no significativa)'
    print(f'\n{nombre}:')
    print(f'  Tendencia:       {res.trend.upper()}')
    print(f'  p-value:         {res.p:.4f} {sig}')
    print(f'  Pendiente Sen:   {res.slope:.5f} {nombre}/año')
    print(f'  Tau de Kendall:  {res.Tau:.4f}')

print('\nINTERPRETACIÓN:')
if res_ndvi.p < 0.05:
    direc = 'CRECIENTE' if res_ndvi.slope > 0 else 'DECRECIENTE'
    print(f'  El NDVI muestra tendencia {direc} significativa ({res_ndvi.slope*100:.3f}% NDVI/año)')
else:
    print('  No hay tendencia estadísticamente significativa en el NDVI.')

## Paso 8 — Exportar resultados

In [ ]:
# Exportar CSV
df.to_csv('serie_temporal_snsm.csv', index=False)
print('CSV exportado: serie_temporal_snsm.csv')

# Exportar mapa de cambio a Google Drive
task = ee.batch.Export.image.toDrive(
    image=diff,
    description='dNDVI_2024_vs_2018_SNSM',
    folder='teledeteccion_maestria',
    fileNamePrefix='dNDVI_2024_vs_2018',
    region=aoi, scale=10, crs='EPSG:4326', maxPixels=1e9
)
task.start()
print(f'Exportación GEE iniciada. Estado: {task.status()["state"]}')
print('Revisa Google Drive en la carpeta "teledeteccion_maestria".')

---

## Preguntas de análisis

1. ¿Hay tendencia significativa en el NDVI? ¿Qué implica para el dosel de la zona cacaotera?
2. ¿El NDRE y el NDVI muestran la misma tendencia? ¿En qué años difieren más?
3. En el mapa dNDVI: ¿dónde hay mayor aumento? ¿Coincide con la reconversión café→cacao?
4. ¿Por qué varía la desviación estándar entre años? ¿Se relaciona con la nubosidad?

## Para investigar (tarea opcional)
- Cambia el filtro a **octubre-noviembre** (temporada húmeda). ¿Cambia la tendencia?
- Usa **MODIS MOD13Q1** (250 m, 16 días). ¿Cómo se compara con Sentinel-2 a 10 m?
- Agrega **Landsat 8/9** para extender la serie hasta 2013.